Solving the same Titanic survival problem as scikit-learn project, but with a neural network 
To compare a classic ML model (Random Forest) against a deep learning approach on the exact same data 

In [23]:
import pandas as pd
import numpy as np
import torch 
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report

torch.manual_seed(42)
df = pd.read_csv("train_cleaned.csv")

In [24]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family size
0,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.2500,NaN,S,2
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38,1,0,PC 17599,71.2833,C85,C,2
2,3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1000,C123,S,2
4,5,0,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.0500,NaN,S,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27,0,0,211536,13.0000,NaN,S,1
887,888,1,1,"Graham, Miss. Margaret Edith",female,19,0,0,112053,30.0000,B42,S,1
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,28,1,2,W./C. 6607,23.4500,NaN,S,4
889,890,1,1,"Behr, Mr. Karl Howell",male,26,0,0,111369,30.0000,C148,C,1


1. Prepare the data (reused same processing idea as scikit-learn)

In [25]:
x = df[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Family size"]]
y = df["Survived"].values

num_cols = ["Age", "Fare", "SibSp", "Parch", "Family size"]
cat_cols = ["Pclass", "Sex", "Embarked"]
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

x_train_t = preprocess.fit_transform(x_train).astype("float32")
x_test_t = preprocess.transform(x_test).astype("float32")
print(x_train_t.shape, x_test_t.shape)


(712, 13) (179, 13)


2. Convert to tensors and build a DataLoader

In [26]:
x_train_t = torch.tensor(x_train_t)
x_test_t = torch.tensor(x_test_t)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(x_train_t, y_train_t), batch_size=16, shuffle=True)

3. Define the Model

In [27]:
class TitanicNet(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.net(x) 

model = TitanicNet(x_train_t.shape[1])
print(model)           

TitanicNet(
  (net): Sequential(
    (0): Linear(in_features=13, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Linear(in_features=16, out_features=2, bias=True)
  )
)


4. Training loop

In [30]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(30):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(xb)

    if epoch % 10 == 0:
        print(f"epoch {epoch}: loss {total_loss/len(train_loader.dataset):.4f}")    


epoch 0: loss 0.6484
epoch 10: loss 0.4127
epoch 20: loss 0.3939


5. Evaluate

In [36]:
model.eval()
with torch.no_grad():
    preds = model(x_test_t).argmax(dim=1)

acc = accuracy_score(y_test_t, preds)
print("Neural Network Test accuracuy:", acc)
print(classification_report(y_test_t, preds))


Neural Network Test accuracuy: 0.8100558659217877
              precision    recall  f1-score   support

           0       0.80      0.92      0.86       110
           1       0.83      0.64      0.72        69

    accuracy                           0.81       179
   macro avg       0.82      0.78      0.79       179
weighted avg       0.81      0.81      0.80       179



6. Save and reload

In [37]:
torch.save(model.state_dict(), "titanic_nn.pt")

loaded = TitanicNet(x_train_t.shape[1])
loaded.load_state_dict(torch.load("titanic_nn.pt"))
loaded.eval()

with torch.no_grad():
    same = torch.allclose(model(x_test_t), loaded(x_test_t))

print("Reloaded model gives identical output:", same)

Reloaded model gives identical output: True
